# Analyse van AskDelphi Relations & Tags endpoint
Dit notebook ondersteunt het verkennen, analyseren en valideren van de Relations & Tags functionaliteit binnen AskDelphi.  
De module biedt helpermethoden voor het toevoegen, beheren en inspecteren van relaties en tags die aan AskDelphi‑topics gekoppeld zijn.  
Het doel van dit notebook is om:

- inzicht te krijgen in de datastructuren en API‑interacties rondom relations & tags
- voorbeelddata te analyseren en patronen te ontdekken
- helpermethoden te testen en gedrag te evalueren
- experimentele of onderzoeksgerichte analyses uit te voeren ter verbetering van de contentstructuur in AskDelphi

Dit notebook vormt daarmee een flexibel startpunt voor verdere exploratie of debugging van de Relations & Tags endpoint.

### Connectie met AskDelphi opzetten

In [ ]:
from ask_delphi_api.authentication import AskDelphiClient
client = AskDelphiClient()
client.authenticate()   # pakt automatisch portal code uit .env

In [ ]:
from ask_delphi_api.project import Project
from ask_delphi_api.topictools import TopicTools
from ask_delphi_api.relation import Relation
from ask_delphi_api.workflow import Workflow
workflow = Workflow(client)
project = Project(client)
topic = TopicTools(client, project)
relation = Relation(client)

### Creatie Digicoach

In [ ]:
import uuid

# Constant
DIGICOACH_NAME = "Digicoach test arie 10feb"
TASK_NAME = "Taak test 10feb"
ACTION_NAME = "Stap test 10feb"
PREDEFINED_SEARCH_NAME = "Voorgedefinieerde zoekopdracht 10feb"

# Create Voorgedefinieerde zoekopdracht topic
topic_id_predefined_search = topic.topic_upload(PREDEFINED_SEARCH_NAME, "Pre-defined search")
topic_version_id_predefined_search = topic.get_topicVersionId(topic_id_predefined_search)
print(f"Created Voorgedefinieerde zoekopdracht topic : {topic_id_predefined_search}")

# Create Digicoach topic
topic_id_digicoach = str(uuid.uuid4())      
topicTitle = DIGICOACH_NAME      
topicTypeId = project.get_topic_type_id("Digitale Coach Procespagina")     
parentTopicId = topic_id_predefined_search
parentTopicRelationTypeId = relation.get_relation_type_id(topic_id_predefined_search, topic_version_id_predefined_search,"Voorgedefinieerde zoekopdracht")
parentTopicVersionId = topic_version_id_predefined_search
relation.add_topic_with_relation(client, topic_id_digicoach, topicTitle, topicTypeId, parentTopicId, parentTopicRelationTypeId, parentTopicVersionId)
print(f"Created Digicoach topic : {topic_id_digicoach}")

# Get Digicoach topic version ID
topic_version_id_digicoach = topic.get_topicVersionId(topic_id_digicoach)

# Tag Digitale Coach Procespagina
relation.add_tag(topic_id_digicoach, topic_version_id_digicoach, "interactie")

# Create Task topic
topic_id_task = str(uuid.uuid4())
topicTitle = TASK_NAME       
topicTypeId = project.get_topic_type_id("Task")     
parentTopicId = topic_id_digicoach
parentTopicRelationTypeId = relation.get_relation_type_id(topic_id_digicoach, topic_version_id_digicoach, "Taak")
parentTopicVersionId = topic_version_id_digicoach
relation.add_topic_with_relation(client, topic_id_task, topicTitle, topicTypeId, parentTopicId, parentTopicRelationTypeId, parentTopicVersionId)
print(f"Created Task topic : {topic_id_task}")

# Get Task topic version ID
topic_version_id_task = topic.get_topicVersionId(topic_id_task)

# Create Action topic
topic_id_action = str(uuid.uuid4())
topicTitle = ACTION_NAME       
topicTypeId = project.get_topic_type_id("Action")     
parentTopicId = topic_id_task
parentTopicRelationTypeId = relation.get_relation_type_id(topic_id_task, topic_version_id_task, "Stap")
parentTopicVersionId = topic_version_id_task
relation.add_topic_with_relation(client, topic_id_action, topicTitle, topicTypeId, parentTopicId, parentTopicRelationTypeId, parentTopicVersionId)
print(f"Created Action topic : {topic_id_action}")

# Checkin Voorgedefinieerde zoekopdracht topic
topic.checkin(topic_id_predefined_search)
print(f"Checked in Voorgedefinieerde zoekopdracht topic : {topic_id_predefined_search}")

# Checkin Digicoach topic
topic.checkin(topic_id_digicoach)
print(f"Checked in Digicoach topic : {topic_id_digicoach}")

# Checkin Digicoach taak topic
topic.checkin(topic_id_task)
print(f"Checked in Digicoach task topic : {topic_id_task}")

# Checkin Digicoach stap topic
topic.checkin(topic_id_action)
print(f"Checked in Digicoach action topic : {topic_id_action}")

# # Delete Task topic
# topic_version_id_task = topic.get_topicVersionId(topic_id_task)
# topic.delete_topic(topic_id_task, topic_version_id_task)
# print(f"Deleted Task topic : {topic_id_task}")

# # Delete Digicoach topic
# topic.delete_topic(topic_id_digicoach, topic_version_id_digicoach)
# print(f"Deleted Digicoach topic : {topic_id_digicoach}")

# # Delete Voorgedefinieerde zoekopdracht topic
# topic.delete_topic(topic_id_predefined_search, topic_version_id_predefined_search)
# print(f"Deleted Voorgedefinieerde zoekopdracht topic : {topic_id_predefined_search}")

### Publiceer Digicoach

In [ ]:
#  Creates a workflow transition request for predefined_search topic.
request_id = workflow.create_workflow_transition_request(topic_id_predefined_search)
transitions_model = workflow.get_workflow_transition_request_transitions_model(request_id)
workflow.update_workflow_transition_request(request_id, transitions_model)
workflow.approve_workflow_transition_request(request_id)

#  Creates a workflow transition request for digicoach topic.
request_id = workflow.create_workflow_transition_request(topic_id_digicoach)
transitions_model = workflow.get_workflow_transition_request_transitions_model(request_id)
workflow.update_workflow_transition_request(request_id, transitions_model)
workflow.approve_workflow_transition_request(request_id)

#  Creates a workflow transition request for task topic.
request_id = workflow.create_workflow_transition_request(topic_id_task)
transitions_model = workflow.get_workflow_transition_request_transitions_model(request_id)
workflow.update_workflow_transition_request(request_id, transitions_model)
workflow.approve_workflow_transition_request(request_id)

### Tag Digicoach

In [ ]:
import json
# Type Digitale Coach Procespagina
endpoint = f"/v1/tenant/{{tenantId}}/project/{{projectId}}/acl/{{aclEntryId}}/topic/{topic_id_digicoach}/tag"

result = client._request("GET", endpoint)
print(json.dumps(result, indent=2, ensure_ascii=False))